In [1]:
import numpy as np
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image

import requests
from io import BytesIO

# --- load test image ---
# img = "dog.png"
# image = np.array(Image.open(img).convert("RGB").resize((224, 224)))
# print(f"Image shape: {image.shape}")  # expected: (224, 224, 3)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
url = 'https://images.unsplash.com/photo-1640384974326-3e72680e0fb3?q=80&w=687&auto=format&fit=crop&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D' # cat 73%
response = requests.get(url)
raw_img = Image.open(BytesIO(response.content)).convert('RGB')
print("Raw image size:",raw_img.size)

# Resize and crop
resize_and_crop = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224)
])

# Convert to tensor and normalize
tensor_and_norm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

resized_img = resize_and_crop(raw_img)

# Input image for the model
input_tensor = tensor_and_norm(resized_img)
input_batch = input_tensor.unsqueeze(0).to(device)
print(input_batch.shape)  # expected: (1, 3, 224, 224)

# --- load resnet-18 ---
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet.eval()

with torch.no_grad():
    predicted_class = resnet(input_batch).argmax(dim=1).item()
    print(f"Predicted class index: {predicted_class}")


Raw image size: (687, 1031)
torch.Size([1, 3, 224, 224])
Predicted class index: 281


In [2]:
from shapiq import ImageExplainer
from shapiq.vision.players import SuperpixelStrategy
from skimage.segmentation import slic

# superpixels = slic(np.array(resized_img), n_segments=20, compactness=10, sigma=1)

player_strategy = SuperpixelStrategy(n_segments=10)
image = input_tensor.numpy().transpose(1, 2, 0) # (H, W, C)
explainer = ImageExplainer(model=resnet, data=image, player_strategy=player_strategy)
interaction_values_resnet = explainer.explain(budget=64)

print(f"n_players: {explainer._n_features}")
print(interaction_values_resnet)

TypeError: ImageImputer expects a ModelArchitecture instance for the model.

In [3]:
from shapiq.vision import ImageExplainer, ClassificationArchitecture, ViTClassificationArchitecture

# --- CNN - resnet example ---
image = input_tensor.numpy().transpose(1, 2, 0)
arch = ClassificationArchitecture(model=resnet)
explainer = ImageExplainer(model=arch, data=image)
iv = explainer.explain_function(x=None, budget=64)

In [8]:
from shapiq.vision import ImageImputer

imputer = ImageImputer(model=arch, image=image)